### RAG Pipeline

In [7]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

def process_all_pdfs(pdf_directory) -> list:
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF Files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error occurred: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")

### Text splitting get into chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

chunks = split_documents(all_pdf_documents)
chunks

Found 6 PDF Files to process

Processing: ipd-cover-letter.pdf
Loaded 1 pages

Processing: olive-tree-cover-letter.pdf
Loaded 1 pages

Processing: thusitha-kithuldora-cv-ipd.pdf
Loaded 3 pages

Processing: thusitha-kithuldora-cv.pdf
Loaded 3 pages

Processing: thusitha-kithuldora-fe-cv.pdf
Loaded 3 pages

Processing: thusitha-kithuldora-se-cv.pdf
Loaded 3 pages

Total documents loaded: 14
Split 14 documents into 40 chunks

Example chunk:
Content: Thusitha Kithuldora, 
Colombo, Sri Lanka. 
 
Hiring Manager, 
IPD Group. 
 
Application for Web Developer Position 
 
Dear Hiring Manager, 
I am writing to express my strong interest in the Web Develo...
Metadata: {'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-07-17T10:58:47+05:30', 'author': 'Thusitha Kithuldora', 'moddate': '2026-07-17T10:58:47+05:30', 'source': '..\\data\\pdf\\ipd-cover-letter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'ipd-cover-letter.pdf', 'file_typ

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-07-17T10:58:47+05:30', 'author': 'Thusitha Kithuldora', 'moddate': '2026-07-17T10:58:47+05:30', 'source': '..\\data\\pdf\\ipd-cover-letter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'ipd-cover-letter.pdf', 'file_type': 'pdf'}, page_content='Thusitha Kithuldora, \nColombo, Sri Lanka. \n \nHiring Manager, \nIPD Group. \n \nApplication for Web Developer Position \n \nDear Hiring Manager, \nI am writing to express my strong interest in the Web Developer position at IPD Group. With over four \nyears of professional frontend development experience and a deep proficiency in modern frameworks like \nReact and Next.js, I am eager to bring my technical skills to a dynamic, 65-year-old industry leader. IPD \nGroup’s reputation as a trusted, user-centric provider of electrical and EV infrastructure solutions strongly \nresonates with my desire to build high-per

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded Successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self,texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated Embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9220.61it/s]


Model loaded Successfully. Embedding dimension: 384


C:\Users\thusi\AppData\Local\Temp\ipykernel_17368\3957891301.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded Successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [23]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document Embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e: 
            print(f"Error initializing vectore store: {e}")
            raise

    def add_documents(self,documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print (f"Adding {len(documents)} documents to vectore store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)): 
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings= embeddings_list,
                metadatas = metadatas,
                documents = documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store = VectorStore()

texts=[doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

### store in the vectore database
vector_store.add_documents(chunks,embeddings)

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0
Generating embeddings for 40 texts...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches: 100%|██████████| 2/2 [00:01<00:00,  1.38it/s]

Generated Embeddings with shape: (40, 384)
Adding 40 documents to vectore store...
Successfully added 40 documents to vector store
Total documents in collection: 40


In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vector_store,embedding_manager)

rag_retriever.retrieve("Thusitha")

Retrieving documents for query: 'Thusitha'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.25it/s]

Generated Embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_e3458d7f_24',
  'content': 'Thusitha Kithuldora \nFrontend Focused Full-Stack Software Engineer \nColombo, Sri Lanka | +94762600331 | thusithakit3@gmail.com \nLinkedIn: thusitha-kithuldora | Portfolio: https://www.thusithakit.com \nProfessional Summary \nResults-driven Software Engineer with a strong foundation in modern programming languages, \nincluding Java and JavaScript. Proven ability to deliver scalable, high -performance web \napplications and bridge hardware and software through IoT deployments. Highly proficient in AI-\nassisted development workflows, utilizing agentic frameworks to accelerate component \ngeneration while critically validating AI -generated code for security, performance, and \nmaintainability. Adept at agile methodologies, defect res olution, and collaborating across teams \nto engineer enterprise solutions. \nTechnical Skills \n• Computer Science Fundamentals: Data Structures, Algorithms, Object-Oriented \nProgramming (OOP), Agile Methodologies